In [ ]:
import os
from epics import PV
import numpy as np
from datetime import datetime

from PIL import Image

In [ ]:
# 定义PV名称列表
pv_names = ["COL-BI:PRF01:CCD_IMAGE"]  # 替换为实际的PV名称
height, width = 1024, 1280  # 固定图像高度和宽度

# 定义保存数据的目录
data_dir = "data"
os.makedirs(data_dir, exist_ok=True)

In [ ]:
def on_image_update(pvname, value, **kwargs):
    """
    PV更新回调函数：当目标PV的图像更新时调用。
    """
    if value is not None:
        # 将图像数据转换为NumPy数组并调整形状
        image_data = np.array(value).reshape((height, width))
        
        # 将图像数据转换为PIL图像对象（灰度图像）
        image = Image.fromarray(image_data.astype(np.uint8), mode='L')
        
        # 生成保存文件名
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        filename = f"{pvname}_{timestamp}.jpg"
        filepath = os.path.join(data_dir, filename)
        
        # 保存图像数据到文件
        image.save(filepath)
        print(f"Saved image from {pvname} to {filepath}")

In [ ]:
# 为每个PV创建PV对象并设置回调函数
pv_objects = []
for pv_name in pv_names:
    pv = PV(pv_name, callback=on_image_update)
    pv_objects.append(pv)

print("Monitoring PVs for image updates... Press Ctrl+C to stop.")

# 保持程序运行
try:
    while True:
        pass
except KeyboardInterrupt:
    print("Stopped monitoring PVs.")